# 05 · Out-of-Sample Validation
#
**Question:** Does an apparent strategy survive on data it was not used to
develop?
#
We use **chronological** validation only — never random shuffling. The
strategies are predefined and frozen. We split the primary seasons into a
development period and a hold-out (out-of-sample) period, and assess whether
performance **persists** on unseen data rather than relying on one strong
historical result.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np

from src import data as D
from src import probabilities as P
from src import strategies as S
from src import backtest as B
from src import metrics as M
from src import plotting as plt

primary = P.add_probability_columns(D.load_processed())


## Chronological train/test split
#
Development (in-sample) period: **2020-21, 2021-22, 2022-23**.
Hold-out (out-of-sample) period: **2023-24, 2024-25, 2025-26**.
The boundary is chosen purely by time — no match in the OOS period was used
to inform or tune the strategies.

In [2]:
dev_seasons = ["20/21", "21/22", "22/23"]
oos_seasons = ["23/24", "24/25", "25/26"]
dev = primary[primary["season"].isin(dev_seasons)]
oos = primary[primary["season"].isin(oos_seasons)]
print("Development rows:", len(dev), "| OOS rows:", len(oos))


Development rows: 1140 | OOS rows: 1140


## Frozen strategies, run on both periods
#
The same strategy objects / parameters are applied unchanged to both periods.

In [3]:
summary = []
for name in S.STRATEGIES:
    m_dev = M.summarize_ledger(B.backtest_strategy(dev, name))
    m_oos = M.summarize_ledger(B.backtest_strategy(oos, name))
    for tag, m in (("dev", m_dev), ("oos", m_oos)):
        m = dict(m); m["period"] = tag; m["label"] = name
        summary.append(m)
tbl = pd.DataFrame(summary)[["label", "period", "bets", "win_rate",
                             "average_odds", "net_profit", "roi",
                             "max_drawdown", "longest_losing_streak"]].round(4)
tbl


,label,period,bets,win_rate,average_odds,net_profit,roi,max_drawdown,longest_losing_streak
0,favourite,dev,1140,0.5342,1.9398,-17.98,-0.0158,33.39,8
1,favourite,oos,1140,0.5491,1.9492,11.46,0.0101,28.16,8
2,favourite_strong,dev,635,0.6205,1.6130,-11.38,-0.0179,18.68,5
3,favourite_strong,oos,637,0.6436,1.6252,8.42,0.0132,14.15,5
4,prob_bucket_45,dev,717,0.6053,1.6649,-10.73,-0.0150,16.38,5
5,prob_bucket_45,oos,708,0.6271,1.6700,7.77,0.0110,19.85,6
6,low_overround,dev,1140,0.5342,1.9398,-17.98,-0.0158,33.39,8
7,low_overround,oos,1137,0.5497,1.9497,12.46,0.0110,28.16,8
8,draw_value_30,dev,253,0.3399,2.9987,4.90,0.0194,25.61,9
9,draw_value_30,oos,276,0.3152,2.9816,-14.78,-0.0536,26.66,10


## Out-of-sample P&L and drawdown

In [4]:
for name in S.STRATEGIES:
    ledger = B.backtest_strategy(oos, name)
    print(f"\n### {name} — out-of-sample")
    fig = plt.cumulative_pnl(ledger)
    fig.show()
    fig = plt.drawdown_curve(ledger)
    fig.show()



### favourite — out-of-sample



### favourite_strong — out-of-sample



### prob_bucket_45 — out-of-sample



### low_overround — out-of-sample



### draw_value_30 — out-of-sample


## Season-by-season out-of-sample ROI

In [5]:
for name in S.STRATEGIES:
    stbl = M.season_summary(B.backtest_strategy(oos, name))
    print(f"\n### {name} (OOS)")
    print(stbl[["season", "bets", "win_rate", "net_profit", "roi"]].round(4).to_string())
    fig = plt.season_roi(stbl)
    fig.show()



### favourite (OOS)
  season  bets  win_rate  net_profit     roi
0  23/24   380    0.5500        3.84  0.0101
1  24/25   380    0.5605        6.91  0.0182
2  25/26   380    0.5368        0.71  0.0019



### favourite_strong (OOS)
  season  bets  win_rate  net_profit     roi
0  23/24   217    0.6452        6.33  0.0292
1  24/25   218    0.6468        0.20  0.0009
2  25/26   202    0.6386        1.89  0.0094



### prob_bucket_45 (OOS)
  season  bets  win_rate  net_profit     roi
0  23/24   240    0.6292        6.08  0.0253
1  24/25   243    0.6214       -4.15 -0.0171
2  25/26   225    0.6311        5.84  0.0260



### low_overround (OOS)
  season  bets  win_rate  net_profit     roi
0  23/24   380    0.5500        3.84  0.0101
1  24/25   380    0.5605        6.91  0.0182
2  25/26   377    0.5385        1.71  0.0045



### draw_value_30 (OOS)
  season  bets  win_rate  net_profit     roi
0  23/24    78    0.3333        0.28  0.0036
1  24/25   113    0.3274       -2.40 -0.0212
2  25/26    85    0.2824      -12.66 -0.1489


## Statistical perspective: ROI confidence
#
A positive ROI is not automatically an edge. We estimate the number of bets
needed and the binomial uncertainty around the win rate to judge whether a
result is plausibly persistent or within sampling noise.

In [6]:
def oos_diagnostics(name, alpha=0.05):
    ledger = B.backtest_strategy(oos, name)
    m = M.summarize_ledger(ledger)
    n = m["bets"]
    if n == 0:
        return {"name": name, "bets": 0}
    p = m["win_rate"]
    se = np.sqrt(p * (1 - p) / n)
    z = 1.96
    win_lo, win_hi = p - z * se, p + z * se
    # break-even win rate at average odds
    be = 1.0 if m["average_odds"] <= 1 else 1.0 / m["average_odds"]
    return {"name": name, "bets": n, "win_rate": round(p, 4),
            "win_ci": (round(win_lo, 4), round(win_hi, 4)),
            "avg_odds": round(m["average_odds"], 3),
            "break_even_winrate": round(be, 4),
            "roi": round(m["roi"], 4)}

print(pd.DataFrame([oos_diagnostics(n) for n in S.STRATEGIES]).to_string())


               name  bets  win_rate            win_ci  avg_odds  break_even_winrate     roi
0         favourite  1140    0.5491   (0.5202, 0.578)     1.949              0.5130  0.0101
1  favourite_strong   637    0.6436  (0.6064, 0.6808)     1.625              0.6153  0.0132
2    prob_bucket_45   708    0.6271  (0.5915, 0.6627)     1.670              0.5988  0.0110
3     low_overround  1137    0.5497  (0.5208, 0.5786)     1.950              0.5129  0.0110
4     draw_value_30   276    0.3152    (0.2604, 0.37)     2.982              0.3354 -0.0536


## Walk-forward persistence of the favourite family
#
To test persistence we incrementally expand the development window and look at
the hold-out season immediately following it ("walk-forward"). We record ROI on
the development window, on the single held-out season, and on the cumulative
hold-out to date.

In [7]:
ordered = ["20/21", "21/22", "22/23", "23/24", "24/25", "25/26"]
rows = []
for cut in range(1, len(ordered) - 1):
    train = ordered[:cut]
    test = [ordered[cut]]
    m_train = M.summarize_ledger(B.backtest_strategy(
        primary[primary["season"].isin(train)], "favourite"))
    m_test = M.summarize_ledger(B.backtest_strategy(
        primary[primary["season"].isin(test)], "favourite"))
    cum_oos = M.summarize_ledger(B.backtest_strategy(
        primary[primary["season"].isin(ordered[cut:])], "favourite"))
    rows.append({"train_until": train[-1], "test_season": test[0],
                 "dev_roi": m_train["roi"], "test_roi": m_test["roi"],
                 "oos_cum_roi": cum_oos["roi"], "bets": m_test["bets"]})
print(pd.DataFrame(rows).round(4))


  train_until test_season  dev_roi  test_roi  oos_cum_roi  bets
0       20/21       21/22  -0.0192   -0.0417       0.0004   380
1       21/22       22/23  -0.0304    0.0136       0.0109   380
2       22/23       23/24  -0.0158    0.0101       0.0101   380
3       23/24       24/25  -0.0093    0.0182       0.0100   380


## Conclusion
#
The central question is whether an odds-only edge **persists** on unseen data.
Based on this dataset:
#
* **Favourite family**: develops on 2020-23 (dev ROI ≈ -1.5%) but turns
  mildly positive out-of-sample (OOS ROI ≈ +1.0% over 2023-26). This is the
  *opposite* of overfitting: the simple, pre-defined rule performs at least as
  well on data it never saw. However the OOS profit is small (~£8–12 over
  ~1,100 bets at £1 stakes), so while not negative, it is **not an
  economically meaningful edge** — it is within plausible sampling variation.
* **draw_value_30** is the clearest non-persistence case: in-sample ROI ≈ +1.9%
  but out-of-sample ROI ≈ **-5.4%**. Any apparent value in backing the draw at
  p >= 0.30 evaporates on unseen data — a textbook illustration of why
  in-sample results alone are not evidence of an edge.
* At unit stakes the absolute profits are tiny; the relevant statistic is
  sign and stability across seasons, not headline ROI.
#
**Result:** None of the tested strategies demonstrates a large, persistent,
economically meaningful odds-only edge. The book's ~5.6% closing overround
absorbs most of the favourite-value signal available from Bet365's La Liga
1X2 closing prices. A mildly positive but small favourite ROI and a clearly
non-persistent draw strategy are the honest headline findings.
